In [1]:
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path
import pandas as pd

In [2]:
caminho_poppler = r"D:/Apps/Release-26.02.0-0/poppler-26.02.0/Library/bin"
pytesseract.pytesseract.tesseract_cmd = r"D:/Program Files/Tesseract-OCR/tesseract.exe"
def extrair_texto_avancado(caminho_pdf):
    print(f"Processando: {caminho_pdf}")
    texto_final_fatura = []
    
    # Configuração Customizada do Tesseract (Direto do README)
    # OEM 3: Usa a rede neural LSTM padrão do Tesseract
    # PSM 6: Assume um bloco de texto uniforme (ótimo para tabelas de faturas)
    config_tesseract = r'--oem 3 --psm 6'
    
    try:
        imagens_paginas = convert_from_path(caminho_pdf, dpi=300, poppler_path=caminho_poppler, timeout=60)
        lista = []
        for num_pagina, imagem in enumerate(imagens_paginas):
            print(f"Lendo Página {num_pagina + 1}...")
            
            # Usando image_to_data (Retorna um DataFrame do Pandas)
            # Timeout de 60 segundos para evitar travamentos
            dados_ocr = pytesseract.image_to_data(
                imagem, 
                lang='por', 
                config=config_tesseract, 
                output_type=Output.DATAFRAME,
                timeout=120
            )

            lista.append(dados_ocr)
            
            # Limpeza rápida: Remover valores vazios e espaços em branco
            dados_ocr = dados_ocr.dropna(subset=['text'])
            dados_ocr = dados_ocr[dados_ocr['text'].str.strip() != '']
            
            # FILTRO DE CONFIANÇA (O pulo do gato para o ByT5)
            # Só pega palavras que o OCR tem mais de 40% de certeza
            # Isso evita mandar "lixo visual" irrecuperável para o ByT5
            palavras_confiaveis = dados_ocr[dados_ocr['conf'] > 40]['text'].tolist()
            
            # Junta as palavras de volta em uma string separada por espaços
            texto_limpo = " ".join(palavras_confiaveis)
            texto_final_fatura.append(texto_limpo)
            
        return {
            "resultado": "\n".join(texto_final_fatura),
            "dados_ocr": lista
        }

    except pytesseract.TesseractError as e:
        return f'Erro: O Tesseract demorou muito e foi interrompido (Timeout).{e}'
    except Exception as e:
        return f"Erro crítico: {e}"



In [3]:
# ==========================================
# TESTANDO
# ==========================================
meu_pdf = "../data/bitmaps/Nubank_2026-06-08.pdf"
dados_ocr = extrair_texto_avancado(meu_pdf)

print("\n--- TEXTO PRONTO PARA O ByT5 ---")
print(dados_ocr['resultado'])

Processando: ../data/bitmaps/Nubank_2026-06-08.pdf
Lendo Página 1...
Lendo Página 2...
Lendo Página 3...
Lendo Página 4...
Lendo Página 5...
Lendo Página 6...

--- TEXTO PRONTO PARA O ByT5 ---
NU Olá, Luiz. Esta é a sua fatura de junho, no valor de R$ 1.462,67 Data de vencimento: O8 JUN 2026 Período vigente: 01 MAI a 01 JUN Limite total do cartão de crédito: R$ 11.200,00
LUIZ HENRIQUE MONTEIRO SILVA DE LIMA FATURA 08 JUN 2026 EMISSÃO E ENVIO 01 JUN 2026 Alternativas de pagamento para a sua Q) Para valores atualizados, consulte o aplicativo. fatura no valor de R$ 1.462,67 1. Pagar o valor total da fatura Você quita o seu saldo, libera seu limite e não paga juros Pagamento total da fatura nem multa. R$ 1.462,67 G/) Sempre a melhor escolha! Não tem juros nem taxas Ainda que pagar a fatura inteira seja a melhor escolha, sabemos que imprevistos acontecem. Aqui, a gente tem opções para não te deixar no atraso. Você pode consultar mais no aplicativo. 2. Parcele a sua fatura Você dá uma entrad

In [4]:
import spacy
nlp = spacy.load("pt_core_news_sm")
document = nlp(dados_ocr['resultado'])

In [7]:
for entidade in document.ents:
    print(f"Texto: {entidade.text} | Categoria: {entidade.label_}")

Texto: NU Olá | Categoria: MISC
Texto: Luiz | Categoria: PER
Texto: R$ 1.462,67 Data | Categoria: MISC
Texto: JUN | Categoria: ORG
Texto: JUN Limite | Categoria: PER
Texto: LUIZ HENRIQUE MONTEIRO SILVA | Categoria: PER
Texto: LIMA | Categoria: LOC
Texto: JUN 2026 | Categoria: MISC
Texto: EMISSÃO | Categoria: MISC
Texto: ENVIO 01 JUN 2026 Alternativas | Categoria: MISC
Texto: Q | Categoria: MISC
Texto: R$ | Categoria: MISC
Texto: Parcele | Categoria: PER
Texto: Válido | Categoria: PER
Texto: Parcelar | Categoria: LOC
Texto: Parcelar | Categoria: LOC
Texto: R$ | Categoria: MISC
Texto: Valor | Categoria: LOC
Texto: R$ | Categoria: MISC
Texto: Valor | Categoria: LOC
Texto: R$ | Categoria: MISC
Texto: R$ | Categoria: MISC
Texto: Ideal | Categoria: LOC
Texto: IOF R$ | Categoria: MISC
Texto: CET | Categoria: ORG
Texto: Faça | Categoria: PER
Texto: R$ | Categoria: MISC
Texto: R$ | Categoria: MISC
Texto: R$ | Categoria: MISC
Texto: IOF | Categoria: MISC
Texto: LUIZ HENRIQUE MONTEIRO SILVA | Cat